# Streaming Structured Output

In [30]:
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from jaxn import JSONParserHandler, StreamingJSONParser
from typing import Any, Dict, Literal
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
import json
import os

In [4]:
load_dotenv()
api_key = os.environ.get("OPENAI_API_KEY")

In [5]:
client = OpenAI(api_key=api_key)

## Streaming Recap

In [6]:
messages = [
    {"role":"user",
     "content": "Tell me about the 1683 Seige of Vienna by the Ottoman Empire."}
]

In [11]:
stream = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    stream=True
)

response = None
for event in stream:
    if hasattr(event, 'delta'):
        print(event.delta, end='')
    if hasattr(event, 'response'):
        response = event.response

The 1683 Siege of Vienna was a pivotal event in European history, marking the climax of centuries of conflict between the Ottoman Empire and the European powers. The siege occurred from July 14 to September 12, 1683, when a large Ottoman army, led by Grand Vizier Kara Mustafa, laid siege to the city of Vienna, which was then a key stronghold of the Habsburg Empire.

### Key Elements of the Siege:

1. **Background**: By the late 17th century, the Ottoman Empire was expanding into Europe. The city of Vienna, strategically situated at the crossroads of Europe, was seen as crucial for further conquests.

2. **The Siege**: The Ottomans surrounded Vienna with a force estimated at 150,000 troops. They employed heavy artillery to bombard the city's fortifications and tried to cut off supplies.

3. **Defense of Vienna**: The city was defended by approximately 15,000 soldiers and approximately 5,000 civilians. Key figures in the defense included Count Ernst Rüdiger von Starhemberg, who organized

## Adding Structured Output

In [13]:
class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in Markdown")

class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]

In [14]:
messages = [
    {
        "role":"user",
        "content": "Tell me about the First Crusade. Discuss what lead up to it, who organized it, what were the important events that marked the First Crusade, and lastly what were the lasting effects of it."
    }
]

In [16]:
try:
    response = client.responses.parse(
        model="gpt-4o-mini-2024-07-18",
        input=messages,
        text_format=ArticleResponse,
        stream=True
    )
except Exception as e:
    print(f"Error: {type(e).__name__} -> {e}")

Error: AttributeError -> 'str' object has no attribute 'output'


In [17]:
with client.responses.stream(
    model="gpt-4o-mini-2024-07-18",
    input=messages,
    text_format=ArticleResponse,
) as stream:
    response = None
    for event in stream:
        if hasattr(event, 'delta'):
            print(event.delta, end='')
        if hasattr(event, 'response'):
            response = event.response

{"title":"The First Crusade: A Turning Point in History","subtitle":"Exploring the Origins, Events, and Lasting Effects of the First Crusade","sections":[{"header":"Background and Causes","text":"The First Crusade, occurring from 1096 to 1099, was rooted in a complex interplay of religious fervor, political ambition, and social unrest in medieval Europe. In the late 11th century, the Byzantine Empire faced increasing threats from Seljuk Turks, particularly after the Battle of Manzikert in 1071, which allowed Muslim forces to encroach on Byzantine territories. Emperor Alexios I Komnenos sought help from the West, leading him to appeal to Pope Urban II.\n\nIn addition to the pleas for military aid, the rise of popular piety and the desire for pilgrimage to the Holy Lands - particularly Jerusalem, which had fallen to Muslim control - fueled the call for a crusade. Urban II, in a bid to unify Christendom and redirect the aggressive tendencies of knights towards a common enemy, called for t

In [18]:
article = response.output_parsed

print('# ' + article.title)
print(article.subtitle)
print()

for section in article.sections:
    print('## ' + section.header)
    print()
    print(section.text)
    print()
    print()

# The First Crusade: A Turning Point in History
Exploring the Origins, Events, and Lasting Effects of the First Crusade

## Background and Causes

The First Crusade, occurring from 1096 to 1099, was rooted in a complex interplay of religious fervor, political ambition, and social unrest in medieval Europe. In the late 11th century, the Byzantine Empire faced increasing threats from Seljuk Turks, particularly after the Battle of Manzikert in 1071, which allowed Muslim forces to encroach on Byzantine territories. Emperor Alexios I Komnenos sought help from the West, leading him to appeal to Pope Urban II.

In addition to the pleas for military aid, the rise of popular piety and the desire for pilgrimage to the Holy Lands - particularly Jerusalem, which had fallen to Muslim control - fueled the call for a crusade. Urban II, in a bid to unify Christendom and redirect the aggressive tendencies of knights towards a common enemy, called for the First Crusade at the Council of Clermont in 1095

## Streaming JSON

In [19]:
raw_json = """
{"title":"The Misadventures of Sparkle","subtitle":"A Tale of Trouble","sections":[{"header":"Once Upon a Time","text":"In a magical land far away, there lived a young unicorn named Sparkle."},{"header":"The Problem","text":"Sparkle had a mischievous streak that often led to chaos."}]}
""".strip()

In [21]:
class ArticleResponseHandler(JSONParserHandler):
    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)


In [22]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)

In [23]:
for c in raw_json:
    parser.parse_incremental(c)

# The Misadventures of Sparkle
A Tale of Trouble


## Once Upon a Time

In a magical land far away, there lived a young unicorn named Sparkle.

## The Problem

Sparkle had a mischievous streak that often led to chaos.

In [24]:
def llm_structured_stream(user_prompt, output_type, parser_handler=JSONParserHandler(), instructions=None, model="gpt-4o-mini"):

    messages = []

    if instructions:
        messages.append({
            "role":"system",
            "content":instructions
        })
    
    messages.append({
        "role":"user",
        "content":user_prompt
    })

    parser = StreamingJSONParser(handler=parser_handler)

    with client.responses.stream(model=model, input=messages, text_format=output_type) as stream:
        response = None
        for event in stream:
            if hasattr(event, 'delta'):
                parser.parse_incremental(event.delta)
            if hasattr(event, 'response'):
                response = event.response
    
    return response

In [25]:
instructions = "your task is to tell the user bed time stories"
user_prompt = "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

# The Enchanted Unicorn
A Magical Bedtime Tale


## Once Upon a Time

In a lush valley far away, where the sun painted the sky with hues of orange and pink, there lived a beautiful unicorn named Luna. Her shimmering white coat glimmered in the light, and her horn sparkled like a thousand stars.

## The Secret of the Valley

The valley was filled with vibrant flowers and whispering trees, but it held a secret that only the creatures of the forest knew. At night, when the moon was full, a magical river would flow with water as clear as crystal and sprinkled with starlight.

## A Brave Heart

One day, a kind-hearted girl named Maya stumbled upon the valley while chasing a colorful butterfly. She was fascinated by the beauty around her and soon discovered Luna drinking from the magical river. Maya approached slowly, her heart racing with excitement.

## An Unlikely Friendship

To her surprise, Luna didn’t run away. Instead, she lowered her head, inviting Maya to come closer. They quickly b

## Streaming Documentation RAG

In [28]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"}
)

files = reader.read()
parsed_docs = [parsed.parse() for parsed in files]

chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)
index = Index(
    text_fields=["title", "content", "description"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def search(query):
    return index.search(query=query, num_results=5)

instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def augment(question, search):
    context = json.dumps(search, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

Indexed 385 chunks from 95 documents


In [31]:
class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [32]:
class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: Dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)

In [33]:
def structured_stream_rag(query):
    search_results = search(query)
    prompt = augment(query, search_results)
    return llm_structured_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )

In [37]:
result = structured_stream_rag("dual booting Ubuntu 24.04 LTS on a Windows machine")

The provided context does not contain any information on dual booting Ubuntu 24.04 LTS on a Windows machine. For guidance on this topic, you may want to refer to the official Ubuntu installation documentation or other resources online.
answer type: reference
follow up question: What are the steps to install Ubuntu 24.04 LTS alongside Windows?
follow up question: How to troubleshoot dual boot issues with Ubuntu and Windows?
follow up question: What software do I need for dual booting?
